In [1]:
import gpboost as gpb
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
# Load data
data = pd.read_csv('C:/Files/GLAN_DATA/first_wave/downscale_data_60/data_with_activity_sequence_all_recod_downsampling_activity_travel.csv')
data = data[data['time_weight'] >= 5]
data = data[data['tree_height'] >= 0]
print(data.shape)
pred_vars = ['household_income', 'age', 'gender', 'education_level', 'employment_status', 'neighborhood_type',
             'work_study', 'housework', 'personal_affair', 'leisure',
             'travel', 'transportation', 'residence', 'industry', 'company', 'shopping',
             'restaurant', 'life_service', 'education_culture', 'entertainment', 'sport_fitness',
             'recreation_tourism', 'healthcare', 'workday', 'time_hour', 'mobility_status',
             'time_nonflexibility', 'home_ornot', 'POI_density', 'POI_diversity', "tree_height"]
# Prepare 5-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=666)
fold_results = []


(8427, 52)


C:\anaconda3\envs\torch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
import pickle
import numpy as np
import pandas as pd
import shap
from tqdm import tqdm


models_dir = 'models'
out_dir = 'shap_outputs'
os.makedirs(out_dir, exist_ok=True)


data = pd.read_csv('C:/Files/GLAN_DATA/first_wave/downscale_data_60/data_with_activity_sequence_all_recod_downsampling_activity_travel.csv')
data = data[data['time_weight'] >= 5]
data = data[data['tree_height'] >= 0]
print(data.shape)


feature_names = ['household_income', 'age', 'gender', 'education_level', 'employment_status', 'neighborhood_type',
                 'work_study', 'housework', 'personal_affair', 'leisure',
                 'travel', 'transportation', 'residence', 'industry', 'company', 'shopping',
                 'restaurant', 'life_service', 'education_culture', 'entertainment', 'sport_fitness',
                 'recreation_tourism', 'healthcare', 'workday', 'time_hour', 'mobility_status',
                 'time_nonflexibility', 'home_ornot', 'POI_density', 'POI_diversity', 'tree_height']


X_data = data[feature_names].values  # Convert to numpy array
X_data_df = pd.DataFrame(X_data, columns=feature_names)


np.random.seed(1111)
val_n = X_data.shape[0]
fixed_sample_indices = np.random.choice(val_n, size=3000, replace=False)
np.save(os.path.join(models_dir, 'selected_index.npy'), fixed_sample_indices)


X_data_sample = X_data[fixed_sample_indices]
X_data_sample_df = pd.DataFrame(X_data_sample, columns=feature_names)

n_features = len(feature_names)


model_files = sorted([f for f in os.listdir(models_dir) if f.startswith('gpb_model_') and f.endswith('.sav')])
n_bootstrap = len(model_files)
print(f"Found {n_bootstrap} models.")


use_approximate = True
n_samples = X_data_sample_df.shape[0]


bootstrap_main_effects = np.empty((n_bootstrap, n_samples, n_features), dtype=np.float32)
interaction_mean_acc = np.zeros((n_samples, n_features, n_features), dtype=np.float64)

for m_idx, mf in enumerate(tqdm(model_files, desc="SHAP CPU")):
    with open(os.path.join(models_dir, mf), 'rb') as f:
        model = pickle.load(f)

    explainer = shap.TreeExplainer(
        model,
        feature_perturbation="tree_path_dependent" if use_approximate else "interventional",
        approximate=use_approximate
    )

    # Compute SHAP values without batching
    inter = explainer.shap_interaction_values(X_data_sample_df)
    main_eff = np.diagonal(inter, axis1=1, axis2=2)
    bootstrap_main_effects[m_idx] = main_eff.astype(np.float32, copy=False)
    interaction_mean_acc += inter.astype(np.float64, copy=False)

    # Clean up
    del explainer, model


interaction_effects_mean = (interaction_mean_acc / n_bootstrap).astype(np.float32)


for j in range(n_features):
    interaction_effects_mean[:, j, j] = 0.0


np.save(os.path.join(out_dir, 'bootstrap_main_effects.npy'), bootstrap_main_effects)
print(f"Saved main effects to {os.path.join(out_dir, 'bootstrap_main_effects.npy')}")

np.save(os.path.join(out_dir, 'interaction_effects_mean.npy'), interaction_effects_mean)
print(f"Saved interaction mean to {os.path.join(out_dir, 'interaction_effects_mean.npy')}")


main_effects_mean = bootstrap_main_effects.mean(axis=0)  # shape: (n_samples, n_features)
main_effects_low = np.percentile(bootstrap_main_effects, 2.5, axis=0)
main_effects_up = np.percentile(bootstrap_main_effects, 97.5, axis=0)

np.save(os.path.join(out_dir, 'main_effects_mean.npy'), main_effects_mean.astype(np.float32))
np.save(os.path.join(out_dir, 'main_effects_low.npy'), main_effects_low.astype(np.float32))
np.save(os.path.join(out_dir, 'main_effects_up.npy'), main_effects_up.astype(np.float32))
print("Saved main effects mean/CI.")

(8427, 52)
Found 1000 models.


SHAP CPU: 100%|██████████| 1000/1000 [10:27:58<00:00, 37.68s/it]


Saved main effects to shap_outputs\bootstrap_main_effects.npy
Saved interaction mean to shap_outputs\interaction_effects_mean.npy
Saved main effects mean/CI.
